# Audit-ready digital twin for lesion response to positive electrostatic charge therapy (PECT)
This notebook reproduces case study 2 of the paper from the digitised data: the twin and its rival models under
leave-one-patient-out (L1) and leave-one-lesion-out (L2), the nested twin ladder with lesion-bootstrap CIs, the per-patient
errors, a Monte Carlo check of the analytic forecast-error variance, the exhaustive within-patient permutation test,
the power simulation, the illustrative cross-species prior and Fig. 2. It writes the two CSV files of digitised data and
`twin_v3_results.json`, from which `revision_analyses.py` draws Fig. 1b. Each analysis has its own random generator.

In [ ]:
import json, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import wilcoxon, spearmanr, norm, beta as beta_dist

SEEDS = dict(boot=101, sim=303, mc=404)   # an independent random generator for each analysis

## 1) Digitised data (Zandi et al. 2023, Front. Med., Fig. 3E = patient 2, Fig. 3G = patient 3; Zandi et al. 2021, Cancer Med., Fig. 3D(ii))
Day 0 was digitised twice in the source figures (end of the pre-treatment series and start of the treatment series).
We use their geometric mean as the single day-0 value and use the pair to estimate digitisation repeatability.

In [ ]:
RAW = [
 # patient, lesion, day-30, day-15, day0_pre, day0_start, day6, day12
 (2, "6 O'Clock", 840, 935, 1040, 1040, 555, 415), (2, "UOQ", 705, 1205, 1520, 1530, 830, 520),
 (2, "UIQ", 350, 440, 535, 540, 345, 80), (2, "Lateral Part", 1215, 1255, 1315, 1315, 1275, 725),
 (2, "Axillary Tail", 390, 550, 590, 590, 510, 315),
 (3, "Retroareolar", 590, 450, 650, 650, 370, 150), (3, "10 O'Clock", 350, 332, 362, 358, 328, 297),
 (3, "9 O'Clock", 303, 275, 333, 333, 215, 131), (3, "8 O'Clock", 343, 318, 347, 349, 284, 312),
 (3, "4 O'Clock", 318, 299, 324, 324, 306, 290), (3, "Far Zone", 179, 146, 192, 193, 173, 60),
]
raw = pd.DataFrame(RAW, columns=["patient", "lesion", "d_m30", "d_m15", "d0_pre", "d0_start", "d6", "d12"])
raw["source_figure"] = np.where(raw.patient == 2, "Zandi 2023 Front Med Fig 3E", "Zandi 2023 Front Med Fig 3G")
raw.to_csv("pect_lesions_digitised.csv", index=False)
ANIMAL = pd.DataFrame([
 ("Control", 0, 0, 200), ("Control", 3, 0, 342), ("Control", 6, 0, 472), ("Control", 9, 0, 631), ("Control", 12, 0, 847),
 ("NECT", 0, -1, 234), ("NECT", 3, -1, 393), ("NECT", 6, -1, 559), ("NECT", 9, -1, 674), ("NECT", 12, -1, 905),
 ("PECT_1kV", 0, 1, 223), ("PECT_1kV", 3, 1, 126), ("PECT_1kV", 6, 1, 40), ("PECT_1kV", 9, 1, 32), ("PECT_1kV", 12, 1, 5),
 ("PECT_2kV", 0, 2, 1106), ("PECT_2kV", 3, 2, 623), ("PECT_2kV", 6, 2, 249), ("PECT_2kV", 9, 2, 126), ("PECT_2kV", 12, 2, 10),
 ("PECT_5kV", 0, 5, 1798), ("PECT_5kV", 3, 5, 479), ("PECT_5kV", 6, 5, 234), ("PECT_5kV", 9, 5, 97), ("PECT_5kV", 12, 5, 25)],
 columns=["group", "day", "kV", "volume_mean"])
ANIMAL["source_figure"] = "Zandi 2021 Cancer Med Fig 3D(ii), group means"
ANIMAL.to_csv("pect_animal_digitised.csv", index=False)

digit_sd = np.sqrt(np.mean((np.log(raw.d0_pre) - np.log(raw.d0_start)) ** 2) / 2)   # SD of one digitised reading
print(f"Digitisation repeatability (duplicate day-0 points): SD of log volume = {digit_sd:.4f}")

T_PRE = np.array([-30., -15., 0.])
W = (T_PRE - T_PRE.mean()) / np.sum((T_PRE - T_PRE.mean()) ** 2)     # OLS slope weights
les = raw[["patient", "lesion"]].copy()
Y = np.log(np.column_stack([raw.d_m30, raw.d_m15, np.sqrt(raw.d0_pre * raw.d0_start), raw.d6, raw.d12]))
les["y_m30"], les["y_m15"], les["y0"], les["y6"], les["y12"] = Y.T
les["g"] = Y[:, :3] @ W
les["k1"] = les.g - (les.y6 - les.y0) / 6
les["k2"] = les.g - (les.y12 - les.y6) / 6
les["dk"] = les.k2 - les.k1
les["V12"] = np.exp(les.y12)
resid = Y[:, :3] - (Y[:, :3].mean(1, keepdims=True) + np.outer(les.g, T_PRE - T_PRE.mean()))
SIGMA = np.sqrt(np.sum(resid ** 2) / (3 * len(les) - 2 * len(les)))
N = len(les)

# exact noise variances implied by the design (shared day-0 scan included)
S2_K1 = SIGMA ** 2 * ((W[2] + 1 / 6) ** 2 + (1 / 6) ** 2 + W[0] ** 2 + W[1] ** 2)
S2_DK = SIGMA ** 2 * (4 + 1 + 1) / 36
print(f"sigma_log = {SIGMA:.4f} (11 df); noise variance of k1 including the shared day-0 scan = {S2_K1:.6f}")
print("Pre-treatment slopes g (per day):", np.round(les.g.values, 4))
rho, prho = spearmanr(les.k1, les.k2)
print(f"Spearman rho(k1,k2) = {rho:.2f}, p = {prho:.2f}")

## 2) Models. All forecasts are made at day 6 for day 12 on the log scale.
Personal-rate options: raw k1 (lesion only), shrunk k1 (Kalman / empirical Bayes), population mean (no personal data).
Drift options: none, or the population mean change in kill rate between intervals. The full twin = shrunk + drift.

In [ ]:
MODELS = {  # key: (label, rate, drift)
 "locf": ("Last observation carried forward", None, None),
 "growth": ("Growth only (no treatment effect)", None, None),
 "static": ("Lesion-only rate (static k)", "raw", False),
 "static_drift": ("Lesion-only rate + drift", "raw", True),
 "pop": ("Population rate, no drift", "pop", False),
 "pop_drift": ("Population rate + drift", "pop", True),
 "twin_nodrift": ("Twin without drift (Kalman only)", "shrunk", False),
 "twin": ("Digital twin (Kalman + drift)", "shrunk", True),
}

def pop_params(train, c_tau=1.0, w_drift=1.0, tau_floor=1e-5):
    n = len(train)
    v_k1 = train.k1.var(ddof=1)
    tau2_raw = v_k1 - S2_K1
    tau2 = max(tau2_raw, tau_floor) * c_tau
    var_d = max(train.dk.var(ddof=1) - S2_DK, 0.0)
    return dict(mu=train.k1.mean(), tau2=tau2, floored=tau2_raw < tau_floor, delta=w_drift * train.dk.mean(),
                var_d=var_d, var_mu=v_k1 / n, var_delta=(w_drift ** 2) * train.dk.var(ddof=1) / n)

def fold_params(train, c_tau=1.0, w_drift=1.0):
    P = pop_params(train, c_tau, w_drift); P["K"] = P["tau2"] / (P["tau2"] + S2_K1); return P

def error_var(Kx, P, drift):
    """Exact variance of (y12 - forecast) on the log scale for a rate shrunk with weight Kx toward the population mean
    (Kx = 1: lesion-only rate; Kx = 0: population rate). Includes the shared day-0 scan, the slope estimate,
    between-lesion rate variance, residual change in rate and plug-in uncertainty of the population mean and drift."""
    c0 = Kx - 6 * (1 - Kx) * W[2]
    v = SIGMA ** 2 * (1 + (1 + Kx) ** 2 + c0 ** 2 + 36 * (1 - Kx) ** 2 * (W[0] ** 2 + W[1] ** 2))
    v += 36 * (1 - Kx) ** 2 * (P["tau2"] + P["var_mu"])
    v += 36 * (P["var_d"] + (P["var_delta"] if drift else 0.0))
    return v

def point(row_y6, row_g, k1, P, key):
    if key == "locf": return row_y6
    if key == "growth": return row_y6 + 6 * row_g
    _, rate, drift = MODELS[key]
    Kx = {"raw": 1.0, "pop": 0.0, "shrunk": P["K"]}[rate]
    kk = P["mu"] + Kx * (k1 - P["mu"])
    return row_y6 + 6 * (row_g - (kk + (P["delta"] if drift else 0.0)))

def predict(train, row, key, c_tau=1.0, w_drift=1.0, k1_obs=None):
    P = fold_params(train, c_tau, w_drift)
    k1 = row.k1 if k1_obs is None else k1_obs
    return point(row.y6, row.g, k1, P, key), P, (P["K"] if MODELS[key][1] == "shrunk" else np.nan)

## 3) Metrics
MdAPE and RMSLE for point forecasts; for probabilistic forecasts on the log scale: 95% coverage, interval width (hi/lo ratio),
Winkler interval score and Gaussian CRPS. Baselines without an analytic predictive distribution receive a Gaussian with SD
equal to the RMS log-error of the same model on the training lesions (empirical predictive).

In [ ]:
def ape(y, z): return np.abs(np.exp(z) - np.exp(y)) / np.exp(y) * 100
def crps_gauss(y, m, s):
    zz = (y - m) / s
    return s * (zz * (2 * norm.cdf(zz) - 1) + 2 * norm.pdf(zz) - 1 / np.sqrt(np.pi))
def interval_score(y, lo, hi, a=0.05):
    return (hi - lo) + 2 / a * (lo - y) * (y < lo) + 2 / a * (y - hi) * (y > hi)

def train_set(proto, i):
    row = les.loc[i]
    if proto == "L1": return les[les.patient != row.patient]
    if proto == "L2": return les.drop(index=i)
    if proto in ("L5",): return les
    raise ValueError

def run(proto, key, c_tau=1.0, w_drift=1.0):
    z, sd, K, fold_tau = np.zeros(N), np.zeros(N), np.full(N, np.nan), []
    for i in range(N):
        tr = train_set(proto, i); row = les.loc[i]
        z[i], P, K[i] = predict(tr, row, key, c_tau, w_drift)
        _, rate, drift = MODELS[key]
        if rate is None:   # empirical predictive SD from the model's log-errors on the training lesions
            sd[i] = np.sqrt(np.mean([(point(tr.loc[j, "y6"], tr.loc[j, "g"], 0, P, key) - tr.loc[j, "y12"]) ** 2 for j in tr.index]))
        else:
            Kx = {"raw": 1.0, "pop": 0.0, "shrunk": P["K"]}[rate]
            sd[i] = np.sqrt(error_var(Kx, P, drift))
        fold_tau.append((P["tau2"], P["floored"]))
    return dict(z=z, sd=sd, K=K, tau=fold_tau)

def summarise(res, y=None):
    y = les.y12.values if y is None else y
    lo, hi = res["z"] - 1.96 * res["sd"], res["z"] + 1.96 * res["sd"]
    cov = np.mean((y >= lo) & (y <= hi))
    return dict(MdAPE=np.median(ape(y, res["z"])), RMSLE=np.sqrt(np.mean((res["z"] - y) ** 2)),
                RMSE=np.sqrt(np.mean((np.exp(res["z"]) - np.exp(y)) ** 2)), coverage=cov * 100,
                width_ratio=np.median(np.exp(hi - lo)), IS=np.mean(interval_score(y, lo, hi)), CRPS=np.mean(crps_gauss(y, res["z"], res["sd"])))

R = {p: {k: run(p, k) for k in MODELS} for p in ["L1", "L2"]}
table = pd.DataFrame([dict(protocol=p, key=k, model=MODELS[k][0], **summarise(R[p][k])) for p in R for k in MODELS])
pd.set_option("display.width", 200)
print("\n=== Honest protocols: all models, both protocols ===")
print(table.drop(columns="key").round(3).to_string(index=False))

for p in ["L1", "L2"]:
    taus = sorted(set((round(t, 6), f) for t, f in R[p]["twin"]["tau"]))
    Ks = R[p]["twin"]["K"]
    print(f"{p}: tau^2 per fold {taus}; Kalman gain range {np.nanmin(Ks):.2f}-{np.nanmax(Ks):.2f}")
k_in = int(np.sum((les.y12 >= R['L1']['twin']['z'] - 1.96 * R['L1']['twin']['sd']) & (les.y12 <= R['L1']['twin']['z'] + 1.96 * R['L1']['twin']['sd'])))
cp = beta_dist.ppf([0.025, 0.975], [k_in, k_in + 1], [N - k_in + 1, N - k_in])
print(f"L1 twin coverage {k_in}/{N}; Clopper-Pearson 95% CI {100*np.nan_to_num(cp[0]):.1f}-{100*(cp[1] if k_in < N else 1):.1f}%")
# L1 error of the twin for each patient separately (a patient-clustered bootstrap is not usable with two patients)
by_patient = {}
for pt in sorted(les.patient.unique()):
    m = (les.patient == pt).values
    by_patient[int(pt)] = dict(lesions=int(m.sum()), MdAPE=float(np.median(ape(les.y12.values[m], R["L1"]["twin"]["z"][m]))))
    print(f"L1 twin MdAPE, patient {pt} ({m.sum()} lesions): {by_patient[int(pt)]['MdAPE']:.1f}%")

## 4) Monte Carlo check of the analytic forecast-error variance

In [ ]:
rng_mc = np.random.default_rng(SEEDS["mc"])
def mc_check(K=0.5, tau2=1e-3, var_d=2e-3, reps=200000):
    g = 0.005; mu = 0.05; delta = 0.03
    k1 = mu + rng_mc.normal(0, np.sqrt(tau2), reps); d = delta + rng_mc.normal(0, np.sqrt(var_d), reps)
    t_all = np.array([-30, -15, 0, 6, 12.])
    z = np.zeros((reps, 5)); z[:, :3] = 6 + g * t_all[:3]
    z[:, 3] = z[:, 2] + 6 * (g - k1); z[:, 4] = z[:, 3] + 6 * (g - (k1 + d))
    y = z + rng_mc.normal(0, SIGMA, (reps, 5))
    gh = y[:, :3] @ W; k1h = gh - (y[:, 3] - y[:, 2]) / 6
    ks = mu + K * (k1h - mu); zhat = y[:, 3] + 6 * (gh - (ks + delta))
    emp = np.var(y[:, 4] - zhat)
    c0 = K - 6 * (1 - K) * W[2]
    ana = SIGMA ** 2 * (1 + (1 + K) ** 2 + c0 ** 2 + 36 * (1 - K) ** 2 * (W[0] ** 2 + W[1] ** 2)) + 36 * (1 - K) ** 2 * tau2 + 36 * var_d
    k1_noise = np.var(k1h - k1)
    return emp, ana, k1_noise
for Kc in [0.0, 0.5, 1.0]:
    emp, ana, kn = mc_check(K=Kc)
    print(f"K={Kc}: forecast-error variance Monte Carlo {emp:.5f} vs analytic {ana:.5f}; k1 noise var MC {kn:.6f} vs exact {S2_K1:.6f}")

## 5) Nested optimism ladder for the twin (same model at every rung) with lesion-bootstrap CIs
L1 leave-one-patient-out -> L2 leave-one-lesion-out -> L3 = L2 with prior scale and drift weight tuned on the evaluation
lesions -> L4 = L3 plus one model variant chosen for ALL lesions by its evaluation error (same definition as case study 1)
-> L5 = the twin assimilates every observation, including day 12 (in-sample state estimate).
Grid for L3: six prior scales (0.25 to 8) x five drift weights (0 to 2). Bootstrap: 4,000 resamples of the 11 lesions, percentile method.

In [ ]:
grid = [(c, w) for c in [0.25, 0.5, 1, 2, 4, 8] for w in [0, 0.5, 1, 1.5, 2]]
def md(res): return np.median(ape(les.y12.values, res["z"]))
L3_scores = {(c, w): md(run("L2", "twin", c, w)) for c, w in grid}
best_cw = min(L3_scores, key=L3_scores.get)
L3 = run("L2", "twin", *best_cw)
cands = {k: run("L2", k, *best_cw) for k in ["static", "static_drift", "pop", "pop_drift", "twin_nodrift", "twin"]}
best_variant = min(cands, key=lambda k: md(cands[k]))
L4 = cands[best_variant]
# L5: in-sample -- population statistics from all lesions and the day-12 observation assimilated into the kill-rate state
def twin_in_sample():
    P = pop_params(les); K = P["tau2"] / (P["tau2"] + S2_K1)
    z = np.zeros(N)
    for i in range(N):
        r = les.loc[i]
        k1s = P["mu"] + K * (r.k1 - P["mu"]); prior_m = k1s + P["delta"]; prior_v = (1 - K) * P["tau2"] + P["var_d"]
        obs_v = SIGMA ** 2 * 2 / 36 + SIGMA ** 2 * np.sum(W ** 2)          # noise of the observed day6->12 rate
        G = prior_v / (prior_v + obs_v); k2s = prior_m + G * (r.k2 - prior_m)
        z[i] = r.y6 + 6 * (r.g - k2s)
    return dict(z=z, sd=np.full(N, np.nan))
L5 = twin_in_sample()

rng_b = np.random.default_rng(SEEDS["boot"])
def boot_md(z, B=4000):
    e = ape(les.y12.values, z); idx = rng_b.integers(0, N, (B, N))
    return np.percentile(np.median(e[idx], axis=1), [2.5, 97.5])
rungs = [("L1", R["L1"]["twin"]), ("L2", R["L2"]["twin"]), ("L3", L3), ("L4", L4), ("L5", L5)]
ladder = pd.DataFrame([dict(rung=n, MdAPE=md(r), CI_lo=boot_md(r["z"])[0], CI_hi=boot_md(r["z"])[1],
                            RMSLE=np.sqrt(np.mean((r["z"] - les.y12.values) ** 2))) for n, r in rungs])
print(f"\nL3 tuned (prior scale, drift weight) = {best_cw}; L4 variant chosen on evaluation error = {best_variant}")
print("=== Nested twin ladder ===\n" + ladder.round(2).to_string(index=False))

## 6) Negative controls and paired tests
Within-patient permutation: the twin receives another lesion's day-6 response from the SAME patient (all within-patient
derangements are enumerated: 44 x 265 = 11,660). Paired Wilcoxon tests ignore clustering and are reported as descriptive only.

In [ ]:
from itertools import permutations
def within_patient_perms():
    groups = [np.where(les.patient == p)[0] for p in sorted(les.patient.unique())]
    per = [[q for q in permutations(g) if all(a != b for a, b in zip(q, g))] for g in groups]
    out = []
    for q2 in per[0]:
        for q3 in per[1]:
            m = np.arange(N); m[groups[0]] = q2; m[groups[1]] = q3; out.append(m)
    return out
perms = within_patient_perms()
FOLDP = {p: [fold_params(train_set(p, i)) for i in range(N)] for p in ["L1", "L2"]}
def twin_with_k1(proto, k1_vec):
    return np.array([point(les.y6[i], les.g[i], k1_vec[i], FOLDP[proto][i], "twin") for i in range(N)])
perm_res = {}
for proto in ["L1", "L2"]:
    real = md(R[proto]["twin"])
    null = np.array([np.median(ape(les.y12.values, twin_with_k1(proto, les.k1.values[m]))) for m in perms])
    perm_res[proto] = dict(real=real, null_median=float(np.median(null)), p=float((1 + np.sum(null <= real)) / (1 + len(null))), n_perm=len(null))
    print(f"{proto}: twin MdAPE {real:.1f}% | within-patient permutation null median {np.median(null):.1f}% | p = {perm_res[proto]['p']:.3f} ({len(null)} derangements)")

def wx(proto, a, b):
    ea = np.abs(R[proto][a]["z"] - les.y12.values); eb = np.abs(R[proto][b]["z"] - les.y12.values)
    return wilcoxon(ea, eb).pvalue
tests = {f"{p}: twin vs {b}": wx(p, "twin", b) for p in ["L1", "L2"] for b in ["static", "static_drift", "pop_drift", "twin_nodrift", "locf"]}
print("Paired Wilcoxon (descriptive, lesions not independent):"); [print(f"  {k}: p = {v:.3f}") for k, v in tests.items()]

## 7) Simulation: can this harness detect a useful twin?
Synthetic cohorts from the fitted generative model. 'Informative': the later kill rate follows the early one (beta = 1).
'Null': the later kill rate is unrelated to the early one (beta = 0). Tests: (a) within-patient permutation of day-6
responses (twin must beat shuffled responses), (b) paired Wilcoxon twin vs population rate. Leave-one-patient-out folds.

In [ ]:
rng_s = np.random.default_rng(SEEDS["sim"])
P_all = pop_params(les)
TRUE = dict(g_mu=les.g.mean(), g_sd=les.g.std(ddof=1), mu=P_all["mu"], tau=np.sqrt(P_all["tau2"]),
            patient_share=0.3, delta=P_all["delta"], sd_d=np.sqrt(P_all["var_d"]))
print("Simulation truth (from the 11 lesions):", {k: round(float(v), 4) for k, v in TRUE.items()})
# scenario: (beta = dependence of later on early kill rate, scale of residual rate change, measurement SD on log scale)
SCEN = {"null": (0.0, 1.0, SIGMA), "as_estimated": (1.0, 1.0, SIGMA), "strong": (1.0, 0.5, SIGMA), "strong_precise": (1.0, 0.5, 0.04)}

def simulate(n_pat, m_les, scen):
    beta, dscale, sig = SCEN[scen]
    pid = np.repeat(np.arange(n_pat), m_les); n = pid.size
    u = rng_s.normal(0, TRUE["tau"] * np.sqrt(TRUE["patient_share"]), n_pat)[pid]
    k1 = TRUE["mu"] + u + rng_s.normal(0, TRUE["tau"] * np.sqrt(1 - TRUE["patient_share"]), n)
    k2 = TRUE["mu"] + TRUE["delta"] + beta * (k1 - TRUE["mu"]) + (1 - beta) * rng_s.normal(0, TRUE["tau"], n) \
         + rng_s.normal(0, TRUE["sd_d"] * dscale, n)
    g = rng_s.normal(TRUE["g_mu"], TRUE["g_sd"], n)
    t = np.array([-30, -15, 0, 6, 12.]); z = np.zeros((n, 5))
    z[:, :3] = np.log(rng_s.uniform(150, 1500, n))[:, None] + np.outer(g, t[:3])
    z[:, 3] = z[:, 2] + 6 * (g - k1); z[:, 4] = z[:, 3] + 6 * (g - k2)
    y = z + rng_s.normal(0, sig, (n, 5)); gh = y[:, :3] @ W
    df = pd.DataFrame(dict(patient=pid, g=gh, y6=y[:, 3], y12=y[:, 4], k1=gh - (y[:, 3] - y[:, 2]) / 6, k2=gh - (y[:, 4] - y[:, 3]) / 6))
    df["dk"] = df.k2 - df.k1
    return df, sig

def lopo_params(df, sig):
    s2k1 = sig ** 2 * ((W[2] + 1 / 6) ** 2 + (1 / 6) ** 2 + W[0] ** 2 + W[1] ** 2)
    mu, K, delta = np.zeros(len(df)), np.zeros(len(df)), np.zeros(len(df))
    for p in np.unique(df.patient):
        te = df.patient.values == p; tr = df[~te]
        tau2 = max(tr.k1.var(ddof=1) - s2k1, 1e-5)
        mu[te], K[te], delta[te] = tr.k1.mean(), tau2 / (tau2 + s2k1), tr.dk.mean()
    return mu, K, delta

def mael(z, y): return np.mean(np.abs(z - y))   # mean absolute log error: the harness's test statistic

def harness_once(n_pat, m_les, scen, n_perm=199):
    df, sig = simulate(n_pat, m_les, scen); y = df.y12.values; y6 = df.y6.values; g = df.g.values; k1 = df.k1.values
    mu, K, delta = lopo_params(df, sig)
    zt = y6 + 6 * (g - (mu + K * (k1 - mu) + delta)); zp = y6 + 6 * (g - (mu + delta))
    real = mael(zt, y)
    groups = [np.where(df.patient.values == p)[0] for p in np.unique(df.patient)]
    null = np.empty(n_perm)
    for b in range(n_perm):
        k1p = k1.copy()
        for ix in groups: k1p[ix] = k1[rng_s.permutation(ix)]
        null[b] = mael(y6 + 6 * (g - (mu + K * (k1p - mu) + delta)), y)
    p_perm = (1 + np.sum(null <= real)) / (1 + n_perm)
    try: p_w = wilcoxon(np.abs(zt - y), np.abs(zp - y), alternative="less").pvalue
    except ValueError: p_w = 1.0
    return p_perm < 0.05, p_w < 0.05

DESIGNS = [(2, 6), (4, 5), (10, 5), (20, 5), (40, 5)]
REPS = 300
sim_rows = []
for n_pat, m in DESIGNS:
    for label in SCEN:
        hits = np.array([harness_once(n_pat, m, label) for _ in range(REPS)])
        sim_rows.append(dict(patients=n_pat, lesions=n_pat * m, scenario=label,
                             permutation_rejection=hits[:, 0].mean() * 100, wilcoxon_rejection=hits[:, 1].mean() * 100))
sim = pd.DataFrame(sim_rows)
print("\n=== Harness rejection rates (% of simulated cohorts): power in informative scenarios, false positives in null ===")
print(sim.round(1).to_string(index=False))

## 8) Illustrative cross-species prior (not a finding). EC50 lies below the lowest tested dose and is not identifiable, so
k(30 kV) is effectively kmax. Predictions include the population drift so they are comparable with the twin.

In [ ]:
g0 = {g: s.volume_mean.iloc[0] for g, s in ANIMAL.groupby("group", sort=False)}
E0 = {g: s.kV.iloc[0] for g, s in ANIMAL.groupby("group", sort=False)}
post = ANIMAL[ANIMAL.day > 0]
def dose_k(E, kmax, ec50): E = np.asarray(E, float); return np.where(E > 0, kmax * E / (ec50 + np.maximum(E, 0)), 0.0)
def sse(eta):
    r, kmax, ec50 = np.exp(eta)
    pred = np.log(post.group.map(g0)) + (r - dose_k(post.group.map(E0), kmax, ec50)) * post.day
    return np.sum((pred - np.log(post.volume_mean)) ** 2)
fit = minimize(sse, np.log([0.1, 0.5, 0.2]), method="Nelder-Mead", options=dict(xatol=1e-10, fatol=1e-10, maxiter=20000))
r_a, kmax_a, ec50_a = np.exp(fit.x); k30 = float(dose_k(30, kmax_a, ec50_a))
print(f"Animal fit (5 group means, 20 post-baseline points): kmax={kmax_a:.3f}/d, EC50={ec50_a:.3f} kV (below lowest dose), k(30 kV)={k30:.3f}/d = {100*k30/kmax_a:.1f}% of kmax")
print(f"kmax (mouse) / median human k1 = {kmax_a/np.median(les.k1):.1f}")
xs = []
for sd_prior in [0.02, 0.05, 0.1, 0.2]:
    z = np.zeros(N)
    for i in range(N):
        r = les.loc[i]; tr = train_set("L1", i); P = pop_params(tr)
        G = sd_prior ** 2 / (sd_prior ** 2 + S2_K1); kk = k30 + G * (r.k1 - k30)
        z[i] = r.y6 + 6 * (r.g - (kk + P["delta"]))
    xs.append(dict(prior_sd=sd_prior, gain=G, MdAPE=np.median(ape(les.y12.values, z))))
xs = pd.DataFrame(xs); print(xs.round(3).to_string(index=False))

## 9) Fig. 2 and saved results
Fig. 1 (both panels) is drawn by `revision_analyses.py` from `casestudy1_results.json` and `twin_v3_results.json`.

In [ ]:
FS = 8; WID, HEI = 3.45, 3.45 * 819 / 1380
plt.rcParams.update({"font.size": FS, "font.family": "DejaVu Sans", "axes.linewidth": 0.7, "xtick.major.width": 0.7, "ytick.major.width": 0.7})
d = R["L1"]["twin"]; o = np.argsort(les.y12.values); xpos = np.arange(N)
lo, hi = np.exp(d["z"] - 1.96 * d["sd"]), np.exp(d["z"] + 1.96 * d["sd"])
fig, ax = plt.subplots(figsize=(WID, HEI))
ax.vlines(xpos, lo[o], hi[o], color="#b7cbe3", lw=3.4, label="twin 95% interval")
ax.scatter(xpos, np.exp(d["z"])[o], color="#1f4e8c", s=13, zorder=3, label="twin forecast")
ax.scatter(xpos, np.exp(R["L1"]["static"]["z"])[o], facecolors="none", edgecolors="#cc5a5a", s=15, lw=0.8, zorder=3, label="lesion-only rate")
outside = (les.y12.values < d["z"] - 1.96 * d["sd"]) | (les.y12.values > d["z"] + 1.96 * d["sd"])
ax.scatter(xpos, les.V12.values[o], color="black", marker="x", s=15, lw=0.9, zorder=4, label="observed")
for j, i in enumerate(o):
    if outside[i]: ax.annotate("outside", (j, les.V12.values[i]), xytext=(3, -9), textcoords="offset points", fontsize=FS - 1.5)
ax.set_yscale("log"); ax.set_xticks(xpos)
ax.set_xticklabels([f"P{les.patient[i]} {les.lesion[i]}" for i in o], rotation=60, ha="right", fontsize=FS - 1.5)
ax.set_ylabel("Day-12 volume (mm$^3$)"); ax.legend(fontsize=FS - 1.5, frameon=False, loc="upper left", ncol=2, handletextpad=0.3, columnspacing=1.0)
ax.set_ylim(25, 8000); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(pad=0.3); plt.savefig("Fig2_600dpi.png", dpi=600); plt.close()

out = dict(digit_sd=digit_sd, sigma=SIGMA, s2_k1=S2_K1, rho=rho, p_rho=prho,
           table=table.to_dict(orient="records"), ladder=ladder.to_dict(orient="records"), best_cw=best_cw, best_variant=best_variant,
           tau={p: sorted(set((float(t), bool(f)) for t, f in R[p]["twin"]["tau"])) for p in R},
           gain={p: [float(np.nanmin(R[p]["twin"]["K"])), float(np.nanmax(R[p]["twin"]["K"]))] for p in R},
           coverage_L1=[k_in, N, float(np.nan_to_num(cp[0])), float(cp[1] if k_in < N else 1)], mdape_L1_by_patient=by_patient,
           permutation=perm_res, wilcoxon=tests, simulation=sim.to_dict(orient="records"),
           animal=dict(kmax=kmax_a, ec50=ec50_a, k30=k30, ratio=kmax_a / np.median(les.k1)), cross_species=xs.to_dict(orient="records"))
json.dump(out, open("twin_v3_results.json", "w"), indent=1, default=float)
print("saved")